# Reddit Scraping for LoL Balance Discussion

## Research Question
To what extent are Reddit discussions about whether specific League of Legends champions need buffs or nerfs associated with the direction of subsequent official balance changes in Patch Notes?

## Data Access Plan (Implemented Here)
- Source: `r/leagueoflegends`
- Scope: defined time window + selected champions
- Filter: balance-related keywords (`buff`, `nerf`, `broken`, `OP`, `weak`, etc.)
- Standardization: champion aliases in Reddit text will be mapped to canonical champion names
- Output: post/comment level dataset that can be linked to Patch Notes by `champion` + `date`

In [ ]:
# Auto-install required packages for this notebook
# This line is just for testing the github action
import sys
import subprocess

REQUIRED_PACKAGES = ["praw", "pandas", "python-dateutil"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED_PACKAGES])

import os
import re
from datetime import datetime, timezone
from typing import Dict, List, Optional, Set

import pandas as pd
import praw

print("Dependencies ready: praw, pandas, python-dateutil")

Dependencies ready: praw, pandas, python-dateutil


In [10]:
# -----------------------------
# Project configuration
# -----------------------------
SUBREDDIT = "leagueoflegends"
START_DATE = "2023-01-01"  # inclusive, UTC day
END_DATE = "2026-12-31"    # inclusive, UTC day

# 选定英雄（可按你的研究样本调整）
SELECTED_CHAMPIONS = [
    "Ahri", "Aatrox", "Akali", "Ashe", "Darius", "Ezreal", "Jinx", "Kaisa", "Yasuo", "Zed"
]

# 英雄别名映射（key=别名，value=标准名）
CHAMPION_ALIASES = {
    "kaisa": "Kaisa",
    "kai'sa": "Kaisa",
    "k'sante": "Ksante",
    "ksante": "Ksante",
    "wukong": "MonkeyKing",
    "monkey king": "MonkeyKing",
}

# balance 相关关键词（先做规则过滤）
BALANCE_KEYWORDS = [
    "buff", "buffed", "nerf", "nerfed", "broken", "op", "overpowered",
    "weak", "underpowered", "too strong", "too weak", "needs buff", "needs nerf"
]

# 方向判定词典（简单词典法，可后续升级成模型）
BUFF_SIGNAL_WORDS = ["buff", "buffed", "needs buff", "too weak", "weak", "underpowered"]
NERF_SIGNAL_WORDS = ["nerf", "nerfed", "needs nerf", "too strong", "broken", "op", "overpowered"]

# 全时间窗建议：先关评论 + 不设帖子上限，直到 START_DATE 自动停止
INCLUDE_COMMENTS = False
MAX_POSTS_TO_SCAN = None            # None = 不设上限（会更慢，但覆盖完整时间窗）
MAX_COMMENTS_PER_POST = 100         # 开评论时可调低到 50-100

start_dt = pd.Timestamp(START_DATE, tz="UTC")
end_dt = pd.Timestamp(END_DATE, tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

selected_set = {c.lower() for c in SELECTED_CHAMPIONS}
print(f"Time window: {start_dt} -> {end_dt}")
print(f"Selected champions: {len(SELECTED_CHAMPIONS)}")

Time window: 2023-01-01 00:00:00+00:00 -> 2026-12-31 23:59:59+00:00
Selected champions: 10


In [11]:
# -----------------------------
# Reddit authentication
# -----------------------------
# Preferred: set env vars first
# REDDIT_CLIENT_ID, REDDIT_CLIENT_SECRET, REDDIT_USER_AGENT
# Optional: REDDIT_USERNAME, REDDIT_PASSWORD

from getpass import getpass

client_id = os.getenv("REDDIT_CLIENT_ID")
client_secret = os.getenv("REDDIT_CLIENT_SECRET")
user_agent = os.getenv("REDDIT_USER_AGENT")

if not client_id:
    client_id = input("REDDIT_CLIENT_ID: ").strip()
if not client_secret:
    client_secret = getpass("REDDIT_CLIENT_SECRET (hidden): ").strip()
if not user_agent:
    user_agent = input("REDDIT_USER_AGENT (e.g. research-scraper/1.0 by u/yourname): ").strip()

if not (client_id and client_secret and user_agent):
    raise ValueError("client_id / client_secret / user_agent cannot be empty")

reddit = praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent=user_agent,
    username=os.getenv("REDDIT_USERNAME"),
    password=os.getenv("REDDIT_PASSWORD"),
    check_for_async=False,
)

print("Authenticated Reddit read client is ready.")

Authenticated Reddit read client is ready.


In [16]:
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "").lower()).strip()


def build_alias_lookup(selected_champions: List[str], alias_dict: Dict[str, str]) -> Dict[str, str]:
    lookup = {}

    # champion 本名本身也纳入可匹配词
    for c in selected_champions:
        base = c.lower().replace("'", "")
        lookup[c.lower()] = c
        lookup[base] = c

    for alias, canonical in alias_dict.items():
        lookup[alias.lower()] = canonical
        lookup[alias.lower().replace("'", "")] = canonical

    return lookup


def extract_champions(text: str, alias_lookup: Dict[str, str], selected_lower_set: Set[str]) -> List[str]:
    t = normalize_text(text)
    t_no_quote = t.replace("'", "")

    found = set()
    for alias, canonical in alias_lookup.items():
        # 词边界匹配，避免误击中长单词
        pattern = rf"(?<![a-z0-9]){re.escape(alias)}(?![a-z0-9])"
        if re.search(pattern, t) or re.search(pattern, t_no_quote):
            # 仅保留研究样本中的英雄
            if canonical.lower() in selected_lower_set:
                found.add(canonical)

    return sorted(found)


def extract_balance_keywords(text: str, keywords: List[str]) -> List[str]:
    t = normalize_text(text)
    hits = []
    for k in keywords:
        if k in t and k not in hits:
            hits.append(k)
    return hits


def has_balance_keywords(text: str, keywords: List[str]) -> bool:
    return len(extract_balance_keywords(text, keywords)) > 0


def infer_balance_direction(text: str) -> str:
    t = normalize_text(text)
    buff_hits = sum(1 for w in BUFF_SIGNAL_WORDS if w in t)
    nerf_hits = sum(1 for w in NERF_SIGNAL_WORDS if w in t)

    if buff_hits > nerf_hits:
        return "buff_request"
    if nerf_hits > buff_hits:
        return "nerf_request"
    if buff_hits == nerf_hits and buff_hits > 0:
        return "mixed"
    return "unclear"

In [17]:
def collect_reddit_balance_discussions(
    reddit_client: praw.Reddit,
    subreddit_name: str,
    start_ts: pd.Timestamp,
    end_ts: pd.Timestamp,
    alias_lookup: Dict[str, str],
    selected_lower_set: Set[str],
    include_comments: bool = True,
    max_posts_to_scan: Optional[int] = None,
    max_comments_per_post: int = 200,
) -> pd.DataFrame:
    subreddit = reddit_client.subreddit(subreddit_name)
    rows = []

    scanned_posts = 0
    kept_posts = 0

    for submission in subreddit.new(limit=max_posts_to_scan):
        scanned_posts += 1
        created = pd.to_datetime(submission.created_utc, unit="s", utc=True)

        if scanned_posts % 200 == 0:
            print(f"Scanned={scanned_posts}, current_post_time={created}")

        # new 流是倒序，早于 start 可以直接停止
        if created < start_ts:
            print(f"Reached start date boundary at post time: {created}")
            break
        if created > end_ts:
            continue

        post_text = f"{submission.title or ''} {submission.selftext or ''}".strip()
        champions = extract_champions(post_text, alias_lookup, selected_lower_set)
        matched_balance_keywords = extract_balance_keywords(post_text, BALANCE_KEYWORDS)

        if champions and matched_balance_keywords:
            direction = infer_balance_direction(post_text)
            for champ in champions:
                rows.append(
                    {
                        "source_type": "post",
                        "subreddit": subreddit_name,
                        "content_id": submission.id,
                        "parent_post_id": submission.id,
                        "champion": champ,
                        "direction": direction,
                        "matched_balance_keywords": "; ".join(matched_balance_keywords),
                        "created_at": created,
                        "title": submission.title,
                        "text": submission.selftext,
                        "score": submission.score,
                        "num_comments": submission.num_comments,
                        "author": str(submission.author) if submission.author else None,
                        "permalink": f"https://www.reddit.com{submission.permalink}",
                    }
                )
            kept_posts += 1

        if include_comments:
            try:
                submission.comments.replace_more(limit=0)
                comments = submission.comments.list()[:max_comments_per_post]
            except Exception:
                comments = []

            for cm in comments:
                c_created = pd.to_datetime(cm.created_utc, unit="s", utc=True)
                if c_created < start_ts or c_created > end_ts:
                    continue

                body = cm.body or ""
                c_champions = extract_champions(body, alias_lookup, selected_lower_set)
                c_matched_balance_keywords = extract_balance_keywords(body, BALANCE_KEYWORDS)

                if not (c_champions and c_matched_balance_keywords):
                    continue

                c_direction = infer_balance_direction(body)
                for champ in c_champions:
                    rows.append(
                        {
                            "source_type": "comment",
                            "subreddit": subreddit_name,
                            "content_id": cm.id,
                            "parent_post_id": submission.id,
                            "champion": champ,
                            "direction": c_direction,
                            "matched_balance_keywords": "; ".join(c_matched_balance_keywords),
                            "created_at": c_created,
                            "title": submission.title,
                            "text": body,
                            "score": cm.score,
                            "num_comments": None,
                            "author": str(cm.author) if cm.author else None,
                            "permalink": f"https://www.reddit.com{submission.permalink}{cm.id}",
                        }
                    )

    df = pd.DataFrame(rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["created_at"], utc=True).dt.floor("D").dt.tz_localize(None)
        df = df.sort_values("created_at", ascending=False).reset_index(drop=True)

    print(f"Scanned posts: {scanned_posts}")
    print(f"Posts matched in window+rules: {kept_posts}")
    print(f"Final rows (post/comment x champion): {len(df)}")

    return df

In [18]:
alias_lookup = build_alias_lookup(SELECTED_CHAMPIONS, CHAMPION_ALIASES)

reddit_balance_df = collect_reddit_balance_discussions(
    reddit_client=reddit,
    subreddit_name=SUBREDDIT,
    start_ts=start_dt,
    end_ts=end_dt,
    alias_lookup=alias_lookup,
    selected_lower_set=selected_set,
    include_comments=INCLUDE_COMMENTS,
    max_posts_to_scan=MAX_POSTS_TO_SCAN,
    max_comments_per_post=MAX_COMMENTS_PER_POST,
)

reddit_balance_df.head(20), len(reddit_balance_df)

Scanned=200, current_post_time=2026-04-10 21:29:56+00:00
Scanned=400, current_post_time=2026-04-08 15:21:14+00:00
Scanned=600, current_post_time=2026-04-06 04:21:02+00:00
Scanned=800, current_post_time=2026-04-03 15:34:06+00:00
Scanned posts: 992
Posts matched in window+rules: 97
Final rows (post/comment x champion): 217


(   source_type        subreddit content_id parent_post_id champion  \
 0         post  leagueoflegends    1sjuvnz        1sjuvnz     Ashe   
 1         post  leagueoflegends    1sjuvnz        1sjuvnz   Ezreal   
 2         post  leagueoflegends    1sjsimf        1sjsimf     Ashe   
 3         post  leagueoflegends    1sjsimf        1sjsimf   Ezreal   
 4         post  leagueoflegends    1sjr6ve        1sjr6ve   Ezreal   
 5         post  leagueoflegends    1sjn8e2        1sjn8e2     Ahri   
 6         post  leagueoflegends    1sjn8e2        1sjn8e2     Ashe   
 7         post  leagueoflegends    1sjn8e2        1sjn8e2   Ezreal   
 8         post  leagueoflegends    1sjmx2x        1sjmx2x   Aatrox   
 9         post  leagueoflegends    1sjkq8j        1sjkq8j   Aatrox   
 10        post  leagueoflegends    1sjkq8j        1sjkq8j    Akali   
 11        post  leagueoflegends    1sjkq8j        1sjkq8j   Ezreal   
 12        post  leagueoflegends    1sjkq8j        1sjkq8j    Kaisa   
 13   

In [15]:
# 导出：原始规则过滤结果 + 聚合结果
raw_out = "reddit_balance_discussions_raw.csv"
agg_out = "reddit_balance_discussions_daily_agg.csv"

reddit_balance_df.to_csv(raw_out, index=False, encoding="utf-8")

# 按 champion + date + direction 聚合，便于后续与 patch notes 关联
if not reddit_balance_df.empty:
    daily_agg_df = (
        reddit_balance_df.groupby(["champion", "date", "direction"], as_index=False)
        .size()
        .rename(columns={"size": "mention_count"})
        .sort_values(["date", "champion", "direction"], ascending=[False, True, True])
    )
else:
    daily_agg_df = pd.DataFrame(columns=["champion", "date", "direction", "mention_count"])

daily_agg_df.to_csv(agg_out, index=False, encoding="utf-8")

print(f"Saved: {raw_out} -> {len(reddit_balance_df)} rows")
print(f"Saved: {agg_out} -> {len(daily_agg_df)} rows")

daily_agg_df.head(20)

Saved: reddit_balance_discussions_raw.csv -> 217 rows
Saved: reddit_balance_discussions_daily_agg.csv -> 92 rows


,champion,date,direction,mention_count
46,Ashe,2026-04-13,nerf_request,1
68,Ezreal,2026-04-13,nerf_request,1
10,Aatrox,2026-04-12,nerf_request,3
21,Ahri,2026-04-12,nerf_request,3
32,Akali,2026-04-12,nerf_request,4
45,Ashe,2026-04-12,nerf_request,7
67,Ezreal,2026-04-12,nerf_request,9
80,Kaisa,2026-04-12,nerf_request,2
9,Aatrox,2026-04-11,nerf_request,2
20,Ahri,2026-04-11,nerf_request,3


## Fallback method (if monthly cloudsearch returns 0)

如果上面的 `cloudsearch + timestamp` 返回 0，使用这个更稳的方案：
- 对每个英雄做宽松搜索（`syntax='plain'`）
- 在 Python 端再按时间窗、关键词、别名做过滤
- 牺牲一点精确性，换取可用结果

In [20]:
def collect_reddit_fallback_search(
    reddit_client: praw.Reddit,
    subreddit_name: str,
    start_ts: pd.Timestamp,
    end_ts: pd.Timestamp,
    alias_lookup: Dict[str, str],
    selected_champions: List[str],
    max_results_per_champion: int = 1000,
) -> pd.DataFrame:
    subreddit = reddit_client.subreddit(subreddit_name)
    rows = []
    seen_posts = set()

    selected_lower = {c.lower() for c in selected_champions}

    for champ in selected_champions:
        # 宽松 query：避免语法导致 0 结果
        query = f'"{champ}" buff OR nerf OR broken OR weak OR "too strong" OR "too weak" OR overpowered OR op'
        print(f"Searching champion={champ}")

        try:
            results = subreddit.search(
                query=query,
                sort="new",
                time_filter="all",
                syntax="plain",
                limit=max_results_per_champion,
            )
        except Exception as e:
            print(f"  search failed: {e}")
            continue

        kept = 0
        for submission in results:
            created = pd.to_datetime(submission.created_utc, unit="s", utc=True)
            if created < start_ts or created > end_ts:
                continue

            if submission.id in seen_posts:
                continue
            seen_posts.add(submission.id)

            post_text = f"{submission.title or ''} {submission.selftext or ''}".strip()
            champions = extract_champions(post_text, alias_lookup, selected_lower)
            if not champions:
                continue

            matched_balance_keywords = extract_balance_keywords(post_text, BALANCE_KEYWORDS)
            if not matched_balance_keywords:
                continue

            direction = infer_balance_direction(post_text)
            for c in champions:
                rows.append(
                    {
                        "source_type": "post",
                        "subreddit": subreddit_name,
                        "content_id": submission.id,
                        "parent_post_id": submission.id,
                        "champion": c,
                        "direction": direction,
                        "matched_balance_keywords": "; ".join(matched_balance_keywords),
                        "created_at": created,
                        "title": submission.title,
                        "text": submission.selftext,
                        "score": submission.score,
                        "num_comments": submission.num_comments,
                        "author": str(submission.author) if submission.author else None,
                        "permalink": f"https://www.reddit.com{submission.permalink}",
                    }
                )
            kept += 1

        print(f"  kept posts in window for {champ}: {kept}")

    df = pd.DataFrame(rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["created_at"], utc=True).dt.floor("D").dt.tz_localize(None)
        df = df.sort_values("created_at", ascending=False).reset_index(drop=True)

    print(f"Final fallback rows: {len(df)}")
    if not df.empty:
        print(f"Date range: {df['created_at'].min()} -> {df['created_at'].max()}")
    return df


# 如果你上一个方法是 0，就运行这个：
reddit_balance_df = collect_reddit_fallback_search(
    reddit_client=reddit,
    subreddit_name=SUBREDDIT,
    start_ts=start_dt,
    end_ts=end_dt,
    alias_lookup=alias_lookup,
    selected_champions=SELECTED_CHAMPIONS,
    max_results_per_champion=1000,
)

reddit_balance_df.head(20), len(reddit_balance_df)

Searching champion=Ahri
  kept posts in window for Ahri: 201
Searching champion=Aatrox
  kept posts in window for Aatrox: 154
Searching champion=Akali
  kept posts in window for Akali: 86
Searching champion=Ashe
  kept posts in window for Ashe: 76
Searching champion=Darius
  kept posts in window for Darius: 131
Searching champion=Ezreal
  kept posts in window for Ezreal: 46
Searching champion=Jinx
  kept posts in window for Jinx: 99
Searching champion=Kaisa
  kept posts in window for Kaisa: 79
Searching champion=Yasuo
  kept posts in window for Yasuo: 78
Searching champion=Zed
  kept posts in window for Zed: 112
Final fallback rows: 2295
Date range: 2025-09-02 04:14:07+00:00 -> 2026-04-13 00:01:46+00:00


(   source_type        subreddit content_id parent_post_id champion  \
 0         post  leagueoflegends    1sjuvnz        1sjuvnz     Ashe   
 1         post  leagueoflegends    1sjuvnz        1sjuvnz   Ezreal   
 2         post  leagueoflegends    1sjsimf        1sjsimf   Ezreal   
 3         post  leagueoflegends    1sjsimf        1sjsimf     Ashe   
 4         post  leagueoflegends    1sjr6ve        1sjr6ve   Ezreal   
 5         post  leagueoflegends    1sjn8e2        1sjn8e2     Ahri   
 6         post  leagueoflegends    1sjn8e2        1sjn8e2   Ezreal   
 7         post  leagueoflegends    1sjn8e2        1sjn8e2     Ashe   
 8         post  leagueoflegends    1sjmx2x        1sjmx2x   Aatrox   
 9         post  leagueoflegends    1sjkq8j        1sjkq8j   Aatrox   
 10        post  leagueoflegends    1sjkq8j        1sjkq8j    Akali   
 11        post  leagueoflegends    1sjkq8j        1sjkq8j   Ezreal   
 12        post  leagueoflegends    1sjkq8j        1sjkq8j    Kaisa   
 13   

## Expand matched posts to full content

基于 `reddit_balance_df` 中命中的帖子，抓取：
- 帖子完整字段（标题、正文、分数、链接、时间等）
- 评论明细（含楼层 depth）

输出两个原始表：
- `reddit_posts_full_content.csv`
- `reddit_comments_full_content.csv`

In [21]:
def expand_full_post_and_comments(
    reddit_client: praw.Reddit,
    matched_df: pd.DataFrame,
    max_comments_per_post: int = 500,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if matched_df is None or matched_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    # post 和 comment 都有 parent_post_id，优先用它定位原帖
    post_ids = (
        matched_df["parent_post_id"].dropna().astype(str).str.replace("t3_", "", regex=False).unique().tolist()
        if "parent_post_id" in matched_df.columns
        else []
    )

    # 兜底：若没有 parent_post_id，则尝试 content_id
    if not post_ids and "content_id" in matched_df.columns:
        post_ids = matched_df["content_id"].dropna().astype(str).str.replace("t3_", "", regex=False).unique().tolist()

    posts_rows = []
    comments_rows = []

    for i, post_id in enumerate(post_ids, start=1):
        if i % 50 == 0:
            print(f"Processed posts: {i}/{len(post_ids)}")

        try:
            submission = reddit_client.submission(id=post_id)
            _ = submission.title  # force fetch
        except Exception as e:
            print(f"Skip post {post_id}: {e}")
            continue

        created = pd.to_datetime(submission.created_utc, unit="s", utc=True)
        posts_rows.append(
            {
                "post_id": submission.id,
                "subreddit": str(submission.subreddit),
                "title": submission.title,
                "selftext": submission.selftext,
                "score": submission.score,
                "upvote_ratio": submission.upvote_ratio,
                "num_comments": submission.num_comments,
                "author": str(submission.author) if submission.author else None,
                "created_at": created,
                "url": submission.url,
                "permalink": f"https://www.reddit.com{submission.permalink}",
                "is_self": submission.is_self,
                "over_18": submission.over_18,
                "link_flair_text": submission.link_flair_text,
            }
        )

        try:
            submission.comments.replace_more(limit=0)
            flat_comments = submission.comments.list()[:max_comments_per_post]
        except Exception:
            flat_comments = []

        for cm in flat_comments:
            c_created = pd.to_datetime(cm.created_utc, unit="s", utc=True)
            comments_rows.append(
                {
                    "post_id": submission.id,
                    "comment_id": cm.id,
                    "parent_id": cm.parent_id,
                    "author": str(cm.author) if cm.author else None,
                    "body": cm.body,
                    "score": cm.score,
                    "created_at": c_created,
                    "depth": getattr(cm, "depth", None),
                    "permalink": f"https://www.reddit.com{submission.permalink}{cm.id}",
                }
            )

    posts_full_df = pd.DataFrame(posts_rows)
    comments_full_df = pd.DataFrame(comments_rows)

    if not posts_full_df.empty:
        posts_full_df = posts_full_df.sort_values("created_at", ascending=False).reset_index(drop=True)
    if not comments_full_df.empty:
        comments_full_df = comments_full_df.sort_values("created_at", ascending=False).reset_index(drop=True)

    return posts_full_df, comments_full_df


posts_full_df, comments_full_df = expand_full_post_and_comments(
    reddit_client=reddit,
    matched_df=reddit_balance_df,
    max_comments_per_post=500,
)

posts_full_df.to_csv("reddit_posts_full_content.csv", index=False, encoding="utf-8")
comments_full_df.to_csv("reddit_comments_full_content.csv", index=False, encoding="utf-8")

print(f"Saved reddit_posts_full_content.csv: {len(posts_full_df)} rows")
print(f"Saved reddit_comments_full_content.csv: {len(comments_full_df)} rows")
posts_full_df.head(10), comments_full_df.head(10)

Processed posts: 50/1062
Processed posts: 100/1062
Processed posts: 150/1062
Processed posts: 200/1062
Processed posts: 250/1062
Processed posts: 300/1062
Processed posts: 350/1062
Processed posts: 400/1062
Processed posts: 450/1062
Processed posts: 500/1062
Processed posts: 550/1062
Processed posts: 600/1062
Processed posts: 650/1062
Processed posts: 700/1062
Processed posts: 750/1062
Processed posts: 800/1062
Processed posts: 850/1062
Processed posts: 900/1062
Processed posts: 950/1062
Processed posts: 1000/1062
Processed posts: 1050/1062
Saved reddit_posts_full_content.csv: 1062 rows
Saved reddit_comments_full_content.csv: 98285 rows


(   post_id        subreddit  \
 0  1sjuvnz  leagueoflegends   
 1  1sjsimf  leagueoflegends   
 2  1sjr6ve  leagueoflegends   
 3  1sjn8e2  leagueoflegends   
 4  1sjmx2x  leagueoflegends   
 5  1sjkq8j  leagueoflegends   
 6  1sjeprv  leagueoflegends   
 7  1sjdk68  leagueoflegends   
 8  1sjc0r4  leagueoflegends   
 9  1sjbzab  leagueoflegends   
 
                                                title  \
 0  Sentinels vs. Cloud9 Kia / LCS 2026 Spring - W...   
 1  Sentinels vs. Cloud9 Kia / LCS 2026 Spring - W...   
 2  Disguised vs. FlyQuest / LCS 2026 Spring - Wee...   
 3  Team Heretics vs. Karmine Corp / LEC 2026 Spri...   
 4                             Im done with this Game   
 5  Team Vitality vs. Shifters / LEC 2026 Spring -...   
 6  Weibo Gaming vs. Bilibili Gaming / LPL 2026 Sp...   
 7  Weibo Gaming vs. Bilibili Gaming / LPL 2026 Sp...   
 8  HANJIN BRION vs. KT Rolster / LCK 2026 Rounds ...   
 9  Oh My God vs. LGD Gaming / LPL 2026 Split 2 - ...   
 
                 